# Cross-Company Relative Position

For any role type you are interested in (`TITLE_PATTERN`): is it paid fairly *relative to its peers at its own company*, and how does that relative position compare across companies?

Each target is placed against salaried roles in the **same seniority band at the same company** (median, IQR, percentile). Company pay level cancels out, so the same role type at a high-paying company and at a lower-paying company can be compared on the same footing.

Peer pools under 5 roles are shown but flagged as thin. Only roles that disclose salary count as peers, and disclosure follows pay-transparency law by location.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# Regex matched against job titles (case-insensitive). Every salaried role that matches,
# at every scraped company, is placed against its own same-seniority peers.
TITLE_PATTERN = r"Technical Writer"
# e.g. r"Instructional Design|Instructional Writer|Curriculum"   r"Forward.Deployed"
COMPANIES = None  # e.g. ["crusoe", "launchdarkly"]; None = all companies in silver

In [ ]:
import db as _db, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from classify import add_usd_salary
from gold_analysis import build_relative_position, data_quality, latest_per_job

_conn = _db.open_silver()
jobs = pd.read_sql("SELECT * FROM jobs", _conn)
_conn.close()
jobs["job_id"] = jobs["job_id"].astype(str)
if COMPANIES:
    jobs = jobs[jobs["company"].isin(COMPANIES)]

rows, unsalaried = [], []
for company, df in jobs.groupby("company"):
    df = latest_per_job(df)
    add_usd_salary(df)
    hits = df[df["title"].str.contains(TITLE_PATTERN, case=False, regex=True, na=False)]
    if hits.empty:
        continue
    print(f"{company}: {data_quality(df)}")
    salaried = df.dropna(subset=["mid_usd"])
    for _, hit in hits.iterrows():
        if hit["job_id"] in salaried["job_id"].values:
            rows.append({"company": company, "job_id": hit["job_id"], **build_relative_position(salaried, hit["job_id"])})
        else:
            unsalaried.append((company, hit["title"]))

positions = pd.DataFrame(rows)
if unsalaried:
    print(f"\nMatched but no disclosed salary (cannot be placed): {len(unsalaried)}")
    for company, title in unsalaried:
        print(f"  {company}: {title}")
print(f"\n{len(positions)} roles matching /{TITLE_PATTERN}/ placed against same-seniority peers")

In [ ]:
# ── Table ───────────────────────────────────────────────────────────
if positions.empty:
    print("No targets with disclosed salary.")
else:
    table = positions.assign(
        target=positions["target_mid"].map("${:,.0f}".format),
        peers=positions.apply(lambda r: f"n={r['n']}{' ⚠ thin' if r['thin'] else ''}", axis=1),
        peer_median=positions["comp_median"].map("${:,.0f}".format),
        peer_iqr=positions.apply(lambda r: f"${r['q1']:,.0f}–${r['q3']:,.0f}", axis=1),
        gap=positions["gap_pct"].map("{:+.1f}%".format),
        pct=positions["percentile"].map("{:.0f}th".format),
    )[["company", "title", "seniority", "location", "target", "peers", "peer_median", "peer_iqr", "gap", "pct"]]
    display(table.style.hide(axis="index"))

In [ ]:
# ── Chart: target vs same-seniority peer IQR ────────────────────────
if not positions.empty:
    fig, ax = plt.subplots(figsize=(11, 1.2 + 0.9 * len(positions)))
    for i, r in positions.iterrows():
        ax.plot([r["q1"], r["q3"]], [i, i], color="#4a7cc9", linewidth=8, alpha=0.5, solid_capstyle="butt")
        ax.plot(r["comp_median"], i, "|", color="#1f3f7a", markersize=22, markeredgewidth=2)
        ax.plot(r["target_mid"], i, "o", color="#d62728", markersize=11, markeredgecolor="black", zorder=5)
        ax.text(max(r["q3"], r["target_mid"]) + 4000, i,
                f"{r['gap_pct']:+.0f}% vs median, n={r['n']}" + (" ⚠" if r["thin"] else ""), va="center", fontsize=9)
    ax.set_yticks(range(len(positions)))
    ax.set_yticklabels([f"{r['company'].title()}: {r['title'][:38]}\n{str(r['location'])[:40]}" for _, r in positions.iterrows()], fontsize=8)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
    ax.set_xlabel("Salary midpoint (USD, annualized)")
    ax.set_title("Target (red) vs same-seniority peers at the same company (band = IQR, tick = median)", fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()